# vLLM vs HF Baseline - Concurrency Lab (SOLUTION)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nukaze/vllm-concurrency-lab-101/blob/main/docs/xd/lablab.ipynb)

> **STEP 0 - ทำก่อนอื่นทั้งหมด:**
> 1. **File > Save a copy in Drive** - บันทึก notebook ลง Google Drive ของคุณก่อน ไม่งั้นถ้า runtime หลุดต้องเริ่มใหม่ทั้งหมด
> 2. **Runtime > Change runtime type > T4 GPU** - vLLM ต้องการ CUDA, CPU/TPU ใช้ไม่ได้

## What you will measure
| Exp | Metric | Method |
|-----|--------|--------|
| A | Single request latency | Send 1 request, measure time-to-completion |
| B | Concurrent throughput | Send N requests simultaneously, measure tokens/sec |
| C | Accuracy diff | Compare outputs from both engines on same prompt |

**Key question:** If the model weights are identical, why does choosing the serving engine
change whether your system handles 10 users or 200 users simultaneously?


In [ ]:
import subprocess

# ตรวจสอบก่อนว่ามี NVIDIA GPU ไหม - vLLM ต้องการ CUDA เท่านั้น ถ้าไม่มีจะ error ทีหลังแบบงง
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type > T4 GPU then reconnect.'
    )
print(result.stdout)

# แสดง CUDA toolkit version - ใช้ตรวจสอบว่า vllm wheel ที่ติดตั้งตรงกับ CUDA ไหม
nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
if nvcc.returncode == 0:
    print(nvcc.stdout)
else:
    print('nvcc not in PATH (nvidia-smi OK -- CUDA runtime available, toolkit PATH not set)')

In [ ]:
!pip install -q uv

# vllm 0.9+ requires CUDA 13 (libcudart.so.13) -- Colab T4 มี CUDA 12 เท่านั้น pin vllm<0.9
# transformers 5.0 ลบ all_special_tokens_extended ออก -- vllm 0.8.x ยังเรียก attribute นี้อยู่
# vllm 0.8.5 ต้องการ transformers>=4.51.1 ดังนั้น pin ให้ครอบทั้งสองด้าน
!uv pip install --system 'vllm<0.9' 'transformers>=4.51.1,<5.0' accelerate openai nest-asyncio

In [ ]:
import os
# ป้องกัน transformers import tensorflow โดยอัตโนมัติ
# vllm ดาวน์เกรด protobuf เป็น version เก่าที่ TF ไม่รองรับ -- เราไม่ได้ใช้ TF เลย
os.environ['USE_TF'] = '0'

import torch
import time
import asyncio
import gc
import nest_asyncio
# Colab มี event loop รันอยู่แล้ว - nest_asyncio ทำให้เราซ้อน async loop ข้างในได้
# ถ้าไม่ apply จะได้ RuntimeError: This event loop is already running
nest_asyncio.apply()

MODEL_ID        = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_NEW_TOKENS  = 150
N_WARMUP        = 1
N_RUNS          = 5
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16]

PROMPTS = [
    'Explain how a CPU works in simple terms.',
    'What is machine learning?',
    'Describe the water cycle.',
    'How does the internet work?',
    'What is photosynthesis?',
    'Explain gravity to a 10-year-old.',
    'What causes earthquakes?',
    'How do vaccines work?',
    'What is DNA?',
    'Explain supply and demand.',
    'How does a car engine work?',
    'What is blockchain?',
    'Describe the solar system.',
    'How do planes fly?',
    'What is climate change?',
    'Explain recursion in programming.',
    'Write simple banking mobile app for me',
    'Explain Quantum Theory for Sci-Math expert'
]
BENCHMARK_PROMPT = PROMPTS[0]

results = {'baseline': {}, 'vllm': {}}
print('Setup complete.')

---
## Section 1 - HF Transformers Baseline
Naive serving: one request processed at a time, no batching optimization.
This is the pattern most tutorials use and what most people start with.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# float16 บังคับใช้เพราะ T4 (Turing architecture) ไม่รองรับ bfloat16
# device_map='auto' ให้ HF จัดการโหลด layer ลง GPU เองอัตโนมัติ
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map='auto'
)
model.eval()
print(f'Model loaded on: {next(model.parameters()).device}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

### Experiment HF A - Single Request Latency

In [ ]:
def baseline_generate(prompt):
    # ต้องใช้ chat template เพื่อให้ baseline และ vLLM ได้รับ prompt format เดียวกัน
    # ถ้าไม่ทำ Exp C จะเทียบผลไม่ได้ เพราะ input จริงๆ ต่างกัน
    messages = [{'role': 'user', 'content': prompt}]
    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text_input, return_tensors='pt').to(model.device)
    with torch.no_grad():
        t0 = time.perf_counter()  # จับเวลาเริ่มต้น ก่อนที่ GPU จะเริ่มคำนวณ

        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)  # รัน autoregressive decoding บน GPU, do_sample=False = greedy (ไม่สุ่ม)

        t1 = time.perf_counter()  # จับเวลาสิ้นสุด หลัง GPU generate ครบทุก token แล้ว

    generated_ids = output[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    latency = t1 - t0
    n_tokens = len(generated_ids)
    return text, latency, n_tokens

# warmup 1 ครั้งก่อนวัดจริง - run แรก GPU ยัง cold อยู่ CUDA kernel ยังไม่ compile
# ถ้าไม่ warmup latency run แรกจะสูงผิดปกติทำให้ค่าเฉลี่ยเบี้ยว
baseline_generate(BENCHMARK_PROMPT)

_log = open('HF-A.log', 'w')
latencies = []
for i in range(N_RUNS):
    text, latency, n_tokens = baseline_generate(BENCHMARK_PROMPT)
    latencies.append(latency)
    tok_s = n_tokens / latency
    print(f'Run {i+1}: {latency:.2f}s | {tok_s:.1f} tok/s')
    _log.write(f'ask#{i+1}: {BENCHMARK_PROMPT}\n')
    _log.write(f'answer#{i+1}: {text}\n')
    _log.write(f'time: {latency:.2f}s | {tok_s:.1f} tok/s\n\n')

results['baseline']['exp_a'] = {
    'latencies': latencies,
    'mean': sum(latencies) / len(latencies),
    'output_sample': text,
}
mean = results['baseline']['exp_a']['mean']
print(f'\nMean latency: {mean:.2f}s')
_log.write(f'--- Summary ---\nMean latency: {mean:.2f}s\n')
_log.close()
print('Saved: HF-A.log')

### Experiment HF B - Concurrent Throughput
Baseline processes requests **sequentially** - this simulates a naive server
with no concurrency support. Watch throughput flatten as N increases.

In [ ]:
baseline_exp_b = {}
_log = open('HF-B.log', 'w')

for n in CONCURRENCY_LEVELS:
    batch = [PROMPTS[i % len(PROMPTS)] for i in range(n)]
    t0 = time.perf_counter()
    total_tokens = 0
    _log.write(f'=== Concurrency {n} ===\n')
    for j, prompt in enumerate(batch):
        _log.write(f'ask#{j+1}: {prompt}\n')
    # loop นี้คือ naive serving - ประมวลผลทีละ request ไม่มี batching
    # จำลองว่า server ธรรมดารับ user 1 คนต่อครั้ง user ที่เหลือรอคิว
    for prompt in batch:
        _, _, n_tok = baseline_generate(prompt)  # เรียก baseline สร้าง output, รับเฉพาะ n_tokens กลับมา (ทิ้ง text และ latency)
        total_tokens += n_tok  # สะสม token รวมของทุก request ใน batch นี้ เพื่อคำนวณ throughput ทีหลัง
    elapsed = time.perf_counter() - t0
    throughput = total_tokens / elapsed
    baseline_exp_b[n] = throughput
    print(f'Concurrency {n:2d} | {throughput:6.1f} tok/s | {elapsed:.1f}s total')
    _log.write(f'throughput: {throughput:.1f} tok/s | elapsed: {elapsed:.1f}s\n\n')

results['baseline']['exp_b'] = baseline_exp_b
_log.write('--- Summary ---\n')
for n, tps in baseline_exp_b.items():
    _log.write(f'Concurrency {n:2d}: {tps:.1f} tok/s\n')
_log.close()
print('\nBaseline Exp B done.')
print('Saved: HF-B.log')

### Free VRAM before starting vLLM
vLLM loads its own copy of the model. We unload the HF model first
to avoid running out of VRAM on T4.


In [ ]:
# ต้อง unload HF model ออกก่อน ไม่งั้น vLLM จะโหลด model ซ้ำแล้ว VRAM เต็ม (OOM)
del model
gc.collect()            # Python garbage collector คืน memory
torch.cuda.empty_cache()  # คืน CUDA caching pool กลับให้ OS

# memory_reserved รวม caching pool ด้วย แม่นกว่า memory_allocated
# ถ้า reserved ยังสูงอยู่ vLLM อาจ OOM ตอน startup
print(f'VRAM reserved after unload: {torch.cuda.memory_reserved()/1e9:.2f} GB')
print(f'VRAM allocated after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## Section 2 - vLLM
vLLM uses **PagedAttention** + **continuous batching**:
- KV cache is managed as non-contiguous blocks (no fragmentation)
- New requests join an in-progress batch immediately (no waiting for a batch to finish)

The client code uses the same OpenAI-compatible API - no client changes needed.


In [ ]:
import requests as req

# เก็บ log ไว้ใน file แทน DEVNULL เพื่อ debug ได้ถ้า server crash
log_file = open('vllm_server.log', 'w')
vllm_proc = subprocess.Popen(
    [
        'vllm', 'serve', MODEL_ID,
        '--port', '8000',
        '--dtype', 'float16',       # T4 ต้องใช้ float16
        '--max-model-len', '2048',  # จำกัด context length ประหยัด KV cache VRAM
        '--gpu-memory-utilization', '0.80',  # จอง VRAM 80% ให้ KV cache เหลือ headroom
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,  # รวม stderr เข้า log file เดียวกัน
)

print('Starting vLLM server', end='', flush=True)
for _ in range(90):
    # poll() คืน None ถ้า process ยังรันอยู่ ถ้าได้ exit code แปลว่า crash แล้ว
    if vllm_proc.poll() is not None:
        log_file.flush()
        raise RuntimeError(
            f'vLLM exited early (code {vllm_proc.returncode}). '
            'Open vllm_server.log in the file browser for details.'
        )
    try:
        if req.get('http://localhost:8000/v1/models', timeout=2).status_code == 200:
            print(' ready!')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(3)
else:
    log_file.flush()
    last_log = open('vllm_server.log').read()[-2000:]
    raise RuntimeError(f'vLLM did not start in time. Last log:\n{last_log}')

### Experiment vLLM A - Single Request Latency

In [ ]:
from openai import OpenAI

client = OpenAI(base_url='http://localhost:8000/v1', api_key='dummy')

def vllm_generate(prompt):
    t0 = time.perf_counter()
    resp = client.chat.completions.create(  # ส่ง HTTP POST ไปยัง vLLM server ที่ localhost:8000 (OpenAI-compatible API)
        model=MODEL_ID,  # ชื่อ model ที่ serve อยู่บน server ต้องตรงกับที่ vLLM โหลดไว้
        messages=[{'role': 'user', 'content': prompt}],  # prompt ในรูปแบบ chat message เหมือน ChatGPT API
        max_tokens=MAX_NEW_TOKENS,  # จำกัด output tokens ให้เท่ากับ baseline เพื่อเทียบกันได้
        temperature=0,  # greedy decoding ไม่สุ่ม ผลลัพธ์ deterministic เหมือน do_sample=False
    )
    latency = time.perf_counter() - t0
    text = resp.choices[0].message.content
    n_tokens = resp.usage.completion_tokens
    return text, latency, n_tokens

# warmup
vllm_generate(BENCHMARK_PROMPT)

_log = open('vLLM-A.log', 'w')
latencies = []
for i in range(N_RUNS):
    text, latency, n_tokens = vllm_generate(BENCHMARK_PROMPT)
    latencies.append(latency)
    tok_s = n_tokens / latency
    print(f'Run {i+1}: {latency:.2f}s | {tok_s:.1f} tok/s')
    _log.write(f'ask#{i+1}: {BENCHMARK_PROMPT}\n')
    _log.write(f'answer#{i+1}: {text}\n')
    _log.write(f'time: {latency:.2f}s | {tok_s:.1f} tok/s\n\n')

results['vllm']['exp_a'] = {
    'latencies': latencies,
    'mean': sum(latencies) / len(latencies),
    'output_sample': text,
}
mean = results['vllm']['exp_a']['mean']
print(f'\nMean latency: {mean:.2f}s')
_log.write(f'--- Summary ---\nMean latency: {mean:.2f}s\n')
_log.close()
print('Saved: vLLM-A.log')

### Experiment vLLM B - Concurrent Throughput
vLLM receives all N requests simultaneously and handles them with continuous batching.
Compare the throughput curve against the baseline.

In [ ]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(base_url='http://localhost:8000/v1', api_key='dummy')

# AsyncOpenAI ส่ง HTTP request แบบ non-blocking ทำให้ await ได้โดยไม่ block thread
async def vllm_generate_async(prompt):
    resp = await async_client.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=MAX_NEW_TOKENS,
        temperature=0,
    )
    return resp.usage.completion_tokens

async def run_concurrent(n):
    batch = [PROMPTS[i % len(PROMPTS)] for i in range(n)]
    t0 = time.perf_counter()
    # asyncio.gather ส่ง request ทุกตัวพร้อมกันในทีเดียว
    # vLLM ฝั่ง server รับทุก request แล้วจัดการผ่าน continuous batching อัตโนมัติ
    token_counts = await asyncio.gather(*[vllm_generate_async(p) for p in batch])  # สร้าง coroutine ทุกตัวแล้วส่งพร้อมกัน, await รอจนทุก request ตอบกลับครบ
    elapsed = time.perf_counter() - t0
    return sum(token_counts) / elapsed, elapsed

async def run_all_concurrent():
    out = {}
    _log = open('vLLM-B.log', 'w')
    for n in CONCURRENCY_LEVELS:
        throughput, elapsed = await run_concurrent(n)
        batch = [PROMPTS[i % len(PROMPTS)] for i in range(n)]
        out[n] = throughput
        print(f'Concurrency {n:2d} | {throughput:6.1f} tok/s | {elapsed:.1f}s total')
        _log.write(f'=== Concurrency {n} ===\n')
        for j, prompt in enumerate(batch):
            _log.write(f'ask#{j+1}: {prompt}\n')
        _log.write(f'throughput: {throughput:.1f} tok/s | elapsed: {elapsed:.1f}s\n\n')
    _log.write('--- Summary ---\n')
    for n, tps in out.items():
        _log.write(f'Concurrency {n:2d}: {tps:.1f} tok/s\n')
    _log.close()
    print('Saved: vLLM-B.log')
    return out

# ใช้ get_event_loop().run_until_complete() แทน asyncio.run()
# เพราะ asyncio.run() จะ close loop หลังจบ ทำให้ AsyncOpenAI client error ใน iteration ถัดไป
vllm_exp_b = asyncio.get_event_loop().run_until_complete(run_all_concurrent())
results['vllm']['exp_b'] = vllm_exp_b
print('\nvLLM Exp B done.')

### Experiment C - Output Accuracy Diff
Same weights, same prompt. Are the outputs identical?
Minor floating-point differences are expected but meaning should be the same.


In [ ]:
from difflib import SequenceMatcher
from datetime import datetime

baseline_out = results['baseline']['exp_a']['output_sample']
vllm_out     = results['vllm']['exp_a']['output_sample']
similarity   = SequenceMatcher(None, baseline_out, vllm_out).ratio()

print(f'Prompt: {BENCHMARK_PROMPT}\n')
print(f'--- Baseline output ---\n{baseline_out}\n')
print(f'--- vLLM output ---\n{vllm_out}\n')
print(f'Similarity score: {similarity:.4f}  (1.0 = identical)')

results['baseline']['exp_c'] = baseline_out
results['vllm']['exp_c']     = vllm_out
results['similarity']        = similarity

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
log_name = f'ExpC_{ts}.log'
with open(log_name, 'w') as f:
    f.write(f'Prompt: {BENCHMARK_PROMPT}\n\n')
    f.write(f'--- Baseline output ---\n{baseline_out}\n\n')
    f.write(f'--- vLLM output ---\n{vllm_out}\n\n')
    f.write(f'Similarity score: {similarity:.4f}\n')
print(f'Saved: {log_name}')

---
## Section 3 - Results


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ORANGE, AZURE = '#f39c12', '#2e86de'

# Exp A - latency bar
ax1 = fig.add_subplot(gs[0, 0])
labels = ['HF Baseline', 'vLLM']
means  = [results['baseline']['exp_a']['mean'], results['vllm']['exp_a']['mean']]
bars   = ax1.bar(labels, means, color=[ORANGE, AZURE], width=0.5, edgecolor='white')
ax1.set_title('Exp A: Single Request Latency', fontweight='bold')
ax1.set_ylabel('Latency (s)')
for b, v in zip(bars, means):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.2f}s', ha='center', fontsize=11)
ax1.set_ylim(0, max(means) * 1.25)

# Exp B - throughput line
ax2 = fig.add_subplot(gs[0, 1])
x   = list(results['baseline']['exp_b'].keys())
ax2.plot(x, list(results['baseline']['exp_b'].values()), 'o-', color=ORANGE, label='HF Baseline', lw=2)
ax2.plot(x, list(results['vllm']['exp_b'].values()),     'o-', color=AZURE,  label='vLLM', lw=2)
ax2.set_title('Exp B: Throughput vs Concurrency', fontweight='bold')
ax2.set_xlabel('Concurrent Users')
ax2.set_ylabel('Throughput (tokens/sec)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Exp A speedup annotation
ax3 = fig.add_subplot(gs[1, 0])
speedup = means[0] / means[1] if means[1] > 0 else 0
ax3.text(0.5, 0.55, f'{speedup:.2f}x faster', ha='center', va='center',
         fontsize=28, fontweight='bold', color=AZURE, transform=ax3.transAxes)
ax3.text(0.5, 0.3, 'vLLM vs Baseline (single request)', ha='center', va='center',
         fontsize=12, color='gray', transform=ax3.transAxes)
ax3.set_title('Exp A: Speedup', fontweight='bold')
ax3.axis('off')

# Exp C similarity (optional -- only if Exp C was run)
ax4 = fig.add_subplot(gs[1, 1])
sim = results.get('similarity')
if sim is not None:
    color = AZURE if sim > 0.95 else ORANGE
    ax4.text(0.5, 0.55, f'Similarity: {sim:.4f}', ha='center', va='center',
             fontsize=22, fontweight='bold', color=color, transform=ax4.transAxes)
    ax4.text(0.5, 0.3, 'Same weights - outputs are near-identical', ha='center', va='center',
             fontsize=11, color='gray', transform=ax4.transAxes)
else:
    ax4.text(0.5, 0.5, 'Exp C not run yet', ha='center', va='center',
             fontsize=16, color='gray', transform=ax4.transAxes)
ax4.set_title('Exp C: Accuracy', fontweight='bold')
ax4.axis('off')

plt.suptitle('vLLM vs HF Baseline - Lab Results (T4 GPU)', fontsize=15, fontweight='bold')
plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_comparison.png')

---
## Cleanup
Stop the vLLM server process when done.


In [ ]:
vllm_proc.terminate()
vllm_proc.wait()
log_file.close()
print('vLLM server stopped.')

---
## Conclusion - What You Actually Learned

### Exp A - Single Request Latency
HF Transformers โหลด model ไว้ใน process เดียวกัน เรียกได้เลย
vLLM เพิ่ม HTTP round-trip ทำให้ latency ที่ N=1 อาจสูงกว่าเล็กน้อย
**Takeaway:** vLLM ไม่ได้ถูกออกแบบมาให้เร็วขึ้นที่ N=1 แต่เพื่อรองรับ N ที่มากขึ้น

### Exp B - Concurrent Throughput
Baseline ประมวลผลทีละ request - throughput แทบไม่เปลี่ยนไม่ว่า N จะเพิ่มแค่ไหน
vLLM รับทุก request พร้อมกันผ่าน continuous batching - throughput เพิ่มตาม N
**Takeaway:** กราฟสองเส้นที่แยกออกจากกันคือเหตุผลหลักที่ production systems ใช้ vLLM

### Exp C - Output Accuracy
Weights เหมือนกัน + prompt เหมือนกัน + temperature=0 = output ควรจะเหมือนกัน
Similarity ~0.97+ บอกว่า engine ต่างกันแต่ผลลัพธ์แทบไม่ต่าง
**Takeaway:** เปลี่ยน serving engine ไม่ได้เปลี่ยน behavior ของ model - migrate ได้โดยไม่ต้องกังวล output drift

### Big Picture
> ถ้า weights เหมือนกัน ทำไม throughput ถึงต่างกัน?

HF: GPU generate เสร็จ 1 request แล้วว่าง รอ request ถัดไป
vLLM: KV cache ถูกจัดการเป็น block ทำให้ request ใหม่เข้า batch ระหว่างที่ request เดิมยังรันอยู่ได้
GPU ไม่ว่าง throughput จึง scale

**Engine เปลี่ยนสิ่งที่ GPU ทำระหว่าง requests - ไม่ใช่สิ่งที่ model คิด**